# 1. Imports

In [ ]:
import sys
import re
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

sys.path.append("../src")
from pombe_feature_functions import CHROMOSOME_END, CENTROMERE_POSITIONS

In [ ]:
genome_length = {
    "I": 5579133,
    "II": 4539804,
    "III": 2452883,
}

# 2. Config

In [ ]:
@dataclass
class Config:
    non_coding_RNA_bed: Path = Path("/data/c/yangyusheng_optimized/DIT_HAP_pipeline/resources/pombase_data/2025-10-01/genome_region/non_coding_rna.bed")
    GtRNAdb_file: Path = Path("/data/c/yangyusheng_optimized/DIT_HAP_pipeline/resources/pombase_data/schiPomb_972H-tRNAs.bed")
    non_coding_RNA_file: Path = Path("/data/c/yangyusheng_optimized/DIT_HAP_pipeline/results/HD_DIT_HAP_generationRAW/19_insertion_in_non_coding_genes/Non_coding_genes_Gene_level_statistics_fitted.tsv")
    output_dir: Path = Path("/data/c/yangyusheng_optimized/DIT_HAP_pipeline/results/HD_DIT_HAP_generationRAW/19_insertion_in_non_coding_genes/analysis_output")
    fasta_file: Path = Path("/data/c/yangyusheng_optimized/DIT_HAP_pipeline/resources/pombase_data/2025-10-01/genome_sequence_and_features/Schizosaccharomyces_pombe_all_chromosomes.fa")

    def __post_init__(self):
        self.non_coding_RNA_meta: pd.DataFrame = pd.read_csv(self.non_coding_RNA_bed, sep="\t")
        self.GtRNAdb: pd.DataFrame = pd.read_csv(self.GtRNAdb_file, sep="\t", header=None, names=["#Chr", "Start", "End", "GtRNAdb_Name", "Score", "Strand", "thickStart", "thickEnd", "itemRgb", "blockCount", "blockSizes", "blockStarts"])[["#Chr", "Start", "End", "GtRNAdb_Name", "Score", "Strand"]].replace(
            {"chrI": "I", "chrII": "II", "chrIII": "III"}
        )
        self.non_coding_RNA_meta = self.non_coding_RNA_meta.merge(
            self.GtRNAdb[["#Chr", "Start", "End", "GtRNAdb_Name"]],
            on=["#Chr", "Start", "End"],
            how="left",
        )

        self.non_coding_RNA: pd.DataFrame = pd.read_csv(self.non_coding_RNA_file, sep="\t")
        self.non_coding_RNA = self.non_coding_RNA_meta[["Systematic ID", "Name", "#Chr", "Start", "Length", "End", "Strand", "Feature", "Type", "GtRNAdb_Name"]].merge(
            self.non_coding_RNA.drop(columns=["Name"]),
            on="Systematic ID",
            how="outer"
        )
        

        self.output_dir.mkdir(parents=True, exist_ok=True)
cfg = Config()

# 3. tRNA analysis

In [ ]:
mRNA_abundance = pd.read_excel("../../resources/Literature/margueratQuantitativeAnalysisFission2012.xlsx", sheet_name="Table_S2", comment="#").set_index('Systematic.name')
mRNA_abundance = mRNA_abundance[['MM1.tot.cpc_ex', 'MM2.tot.cpc_ex', 'MN1.tot.cpc_ex', 'MN2.tot.cpc_ex']].copy()
mRNA_abundance.columns = pd.MultiIndex.from_tuples([("EMM_Proliferating_Cell_RNA_Abundance", "replicate1"), ("EMM_Proliferating_Cell_RNA_Abundance", "replicate2"), ("EMM_Nitrogen_Starved_Cell_RNA_Abundance", "replicate1"), ("EMM_Nitrogen_Starved_Cell_RNA_Abundance", "replicate2")], name=["Condition", "Replicate"])
mean_mRNA_abundance = mRNA_abundance.T.groupby(level="Condition").mean().T
std_mRNA_abundance = mRNA_abundance.T.groupby(level="Condition").std().T
cv_mRNA_abundance = std_mRNA_abundance / mean_mRNA_abundance
mRNA_abundance_statistics = pd.concat([mean_mRNA_abundance, std_mRNA_abundance, cv_mRNA_abundance], axis=1, keys=['mean', 'std', 'cv'])
mRNA_abundance_statistics.columns = [ f"{col[0]}_{col[1]}" for col in mRNA_abundance_statistics.columns]

In [ ]:
def extract_tRNA_amino_acid_and_anticodon(row):
    sysID_tRNA = row["Systematic ID"]
    tRNA_name = row["GtRNAdb_Name"]

    amino_acid_match = re.search(r"TRNA(\w+)\.", sysID_tRNA).group(1) if pd.notna(tRNA_name) else None
    anticodon_match = tRNA_name.split("-")[2] if pd.notna(tRNA_name) else None

    return pd.Series({"Amino_Acid": amino_acid_match, "Anticodon": anticodon_match})

def location_category(row, CHROMOSOME_END, CENTROMERE_POSITIONS):
    chrom = row["#Chr"]
    start = row["Start"]
    end = row["End"]
    
    # Get centromere boundaries
    centromere_start, centromere_end = CENTROMERE_POSITIONS.get(chrom, (None, None))

    midpoint = (start + end) // 2
    # telemere and centromere related features
    abs_distance_from_telomere = min(abs(midpoint - CHROMOSOME_END["left"].get(chrom, np.nan)), abs(midpoint - CHROMOSOME_END["right"].get(chrom, np.nan)))
    relative_distance_from_telomere = round(abs_distance_from_telomere / genome_length[chrom], 3)
    abs_distance_from_centromere = min(abs(midpoint - centromere_start), abs(midpoint - centromere_end))
    relative_distance_from_centromere = round(abs_distance_from_centromere / genome_length[chrom], 3)
    
    # Check if in centromere
    if centromere_start and centromere_end:
        if (start >= centromere_start and start <= centromere_end) or (end >= centromere_start and end <= centromere_end):
            return "centromere", abs_distance_from_telomere, relative_distance_from_telomere, 0, 0
    
    # Get telomere boundaries
    left_telomere = CHROMOSOME_END["left"].get(chrom)
    right_telomere = CHROMOSOME_END["right"].get(chrom)
    
    # Check if in telomere
    if left_telomere and start <= left_telomere:
        return "telomere", abs_distance_from_telomere, relative_distance_from_telomere, abs_distance_from_centromere, relative_distance_from_centromere
    if right_telomere and end >= right_telomere:
        return "telomere", abs_distance_from_telomere, relative_distance_from_telomere, abs_distance_from_centromere, relative_distance_from_centromere
    
    return "other", abs_distance_from_telomere, relative_distance_from_telomere, abs_distance_from_centromere, relative_distance_from_centromere

all_nuclear_tRNAs = cfg.non_coding_RNA.query("Feature == 'tRNA' and `#Chr` != 'mitochondrial'").copy()
all_nuclear_tRNAs[["Amino_Acid", "Anticodon"]] = all_nuclear_tRNAs.apply(extract_tRNA_amino_acid_and_anticodon, axis=1)
all_nuclear_tRNAs["tRNA_copy_number"] = all_nuclear_tRNAs.groupby(["Amino_Acid", "Anticodon"])["Systematic ID"].transform("count")
all_nuclear_tRNAs[["Location", "abs_distance_from_telomere", "relative_distance_from_telomere", "abs_distance_from_centromere", "relative_distance_from_centromere"]] = all_nuclear_tRNAs.apply(location_category, axis=1, CHROMOSOME_END=CHROMOSOME_END, CENTROMERE_POSITIONS=CENTROMERE_POSITIONS, result_type="expand")
all_nuclear_tRNAs = all_nuclear_tRNAs.sort_values(["tRNA_copy_number", "um", "Location"], ascending=[True, False, True])
all_nuclear_tRNAs = all_nuclear_tRNAs.merge(
    mRNA_abundance_statistics,
    left_on="Systematic ID",
    right_index=True,
    how="left"
)

analyzed_tRNAs = all_nuclear_tRNAs.query("um.notna()").copy().sort_values(["tRNA_copy_number","um"], ascending=[True, False])

# analyzed_tRNAs[["Amino_Acid", "Anticodon"]] = analyzed_tRNAs.apply(extract_tRNA_amino_acid_and_anticodon, axis=1)

print(f"Total nuclear tRNAs: {len(all_nuclear_tRNAs)}")
print(f"Analyzed tRNAs: {len(analyzed_tRNAs)}")
print(f"Percentage of nuclear tRNAs analyzed: {len(analyzed_tRNAs) / len(all_nuclear_tRNAs) * 100:.2f}%")

In [ ]:
all_nuclear_tRNAs["Location"].value_counts()

In [ ]:
single_copy_tRNAs = all_nuclear_tRNAs.query("tRNA_copy_number == 1")

In [ ]:
single_copy_tRNAs

In [ ]:
all_nuclear_tRNAs.to_excel(cfg.output_dir / "all_nuclear_tRNAs.xlsx", index=False)

In [ ]:
from Bio import SeqIO
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# Index the genome fasta for quick access
chrom_index = SeqIO.index(str(cfg.fasta_file), "fasta")

UPSTREAM_LEN = 200
DOWNSTREAM_LEN = 200
NUC_MAP = {"A": 1, "C": 2, "G": 3, "T": 4, "N": 0, "a": 1, "c": 2, "g": 3, "t": 4, "n": 0}

# Custom colormap: N=white, A=red, C=green, G=yellow, T=blue
NUC_COLORS = ["#ffffff", "#e41a1c", "#4daf4a", "#ffff33", "#377eb8"]
nuc_cmap = ListedColormap(NUC_COLORS)
nuc_bounds = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], nuc_cmap.N)

def get_flanking_sequences(tRNA_row: pd.Series) -> tuple:
    """Return (upstream_seq, downstream_seq) as strings of length 200 each."""
    chrom = str(tRNA_row["#Chr"])
    start = int(tRNA_row["Start"])
    end = int(tRNA_row["End"])
    strand = tRNA_row["Strand"]

    # Fetch the whole chromosome sequence
    chrom_seq = chrom_index[chrom]
    seq_len = len(chrom_seq)

    if strand == "+":
        # Upstream: before the start
        up_start = max(0, start - UPSTREAM_LEN)
        up_end = start  # up_end is start position (1‑based)
        up_seq = chrom_seq[up_start:up_end].seq
        if up_start == 0:
            up_seq = "N" * (UPSTREAM_LEN - len(up_seq)) + str(up_seq)
        else:
            up_seq = str(up_seq)

        # Downstream: after the end
        dn_start = end
        dn_end = min(seq_len, end + DOWNSTREAM_LEN)
        dn_seq = chrom_seq[dn_start:dn_end].seq
        if dn_end == seq_len:
            dn_seq = str(dn_seq) + "N" * (DOWNSTREAM_LEN - len(dn_seq))
        else:
            dn_seq = str(dn_seq)
    else:  # strand == "-"
        # For minus strand, upstream is the reverse complement of region after the end
        dn_start = end
        dn_end = min(seq_len, end + UPSTREAM_LEN)
        up_seq = chrom_seq[dn_start:dn_end].seq.reverse_complement()
        if dn_end == seq_len:
            up_seq = str(up_seq) + "N" * (UPSTREAM_LEN - len(up_seq))
        else:
            up_seq = str(up_seq)

        # Downstream is the reverse complement of region before the start
        up_start = max(0, start - DOWNSTREAM_LEN)
        up_end = start
        dn_seq = chrom_seq[up_start:up_end].seq.reverse_complement()
        if up_start == 0:
            dn_seq = "N" * (DOWNSTREAM_LEN - len(dn_seq)) + str(dn_seq)
        else:
            dn_seq = str(dn_seq)

    return up_seq, dn_seq

# Use analyzed_tRNAs (with fitted data) or all_nuclear_tRNAs
tRNAs_to_plot = single_copy_tRNAs  # change to all_nuclear_tRNAs if needed

upstream_seqs = []
downstream_seqs = []
valid_indices = []

for idx, row in tRNAs_to_plot.iterrows():
    try:
        up, dn = get_flanking_sequences(row)
        if len(up) == UPSTREAM_LEN and len(dn) == DOWNSTREAM_LEN:
            upstream_seqs.append(up)
            downstream_seqs.append(dn)
            valid_indices.append(idx)
    except Exception as e:
        # skip problematic rows
        continue

# Get GtRNAdb_Name labels for valid tRNAs in the same order
tRNA_labels = tRNAs_to_plot.loc[valid_indices, "GtRNAdb_Name"].tolist()

# Convert sequences to numeric matrices
def seq_to_matrix(seqs: list) -> np.ndarray:
    mat = np.zeros((len(seqs), len(seqs[0])), dtype=int)
    for i, seq in enumerate(seqs):
        for j, nuc in enumerate(seq):
            mat[i, j] = NUC_MAP.get(nuc, 0)
    return mat

up_mat = seq_to_matrix(upstream_seqs)
dn_mat = seq_to_matrix(downstream_seqs)

# Plot heatmaps
fig, axes = plt.subplots(2, 1, figsize=(14, max(8, len(valid_indices) * 0.15)))

# Upstream heatmap
im0 = axes[0].imshow(up_mat, aspect="auto", cmap=nuc_cmap, norm=nuc_bounds, interpolation="nearest")
axes[0].set_title(f"Upstream {UPSTREAM_LEN} bp (n={len(valid_indices)})")
axes[0].set_xlabel("Position relative to start")
axes[0].set_ylabel("tRNA genes")
axes[0].set_xticks(list(range(0, UPSTREAM_LEN - 1,10)) + [UPSTREAM_LEN - 1])
axes[0].set_xticklabels(list(range(0, UPSTREAM_LEN - 1,10)) + [UPSTREAM_LEN - 1])
axes[0].set_yticks(range(len(tRNA_labels)))
axes[0].set_yticklabels(tRNA_labels, fontsize=6)
cbar0 = fig.colorbar(im0, ax=axes[0], ticks=[0, 1, 2, 3, 4])
cbar0.ax.set_yticklabels(["N", "A", "C", "G", "T"])
cbar0.set_label("Nucleotide")

# Downstream heatmap
im1 = axes[1].imshow(dn_mat, aspect="auto", cmap=nuc_cmap, norm=nuc_bounds, interpolation="nearest")
axes[1].set_title(f"Downstream {DOWNSTREAM_LEN} bp (n={len(valid_indices)})")
axes[1].set_xlabel("Position relative to end")
axes[1].set_ylabel("tRNA genes")
axes[1].set_xticks(list(range(0, DOWNSTREAM_LEN - 1,10)) + [DOWNSTREAM_LEN - 1])
axes[1].set_xticklabels(list(range(0, DOWNSTREAM_LEN - 1,10)) + [DOWNSTREAM_LEN - 1])
axes[1].set_yticks(range(len(tRNA_labels)))
axes[1].set_yticklabels(tRNA_labels, fontsize=6)
cbar1 = fig.colorbar(im1, ax=axes[1], ticks=[0, 1, 2, 3, 4])
cbar1.ax.set_yticklabels(["N", "A", "C", "G", "T"])
cbar1.set_label("Nucleotide")

plt.tight_layout()
plt.show()
plt.close()